# 🎓 Foundry Client desde cero — construcción guiada

**Objetivo**: construir `foundry_client.py` desde la primera línea, entendiendo cada decisión de diseño. Cuando termines este notebook, vas a poder escribir los próximos 4 embedders (OpenAI directo, BGE-M3, Voyage, Jina) con los ojos cerrados — porque el patrón es el mismo, solo cambia el SDK.

## ¿Qué vamos a construir?

Un wrapper limpio sobre el SDK `openai` que conecta a Microsoft Foundry y produce vectores de embeddings. La interfaz final será así de simple:

```python
embedder = FoundryEmbedder()
vectors = embedder.embed(["chunk 1", "chunk 2"])
# vectors.shape → (2, 1536)
```

## ¿Qué vas a aprender?

1. **Configuración limpia**: leer credenciales desde `.env` con validación temprana
2. **Inmutabilidad con `frozen=True`**: por qué un config no debe mutar
3. **El patrón Embedder**: una interfaz mínima y reutilizable
4. **Fail-fast**: cómo dar errores útiles en vez de crashes a las 2 horas
5. **SDK choice**: por qué `openai` y no `azure-ai-inference` (deprecation, routing, drop-in)

## Estructura del notebook

| Sección | Construimos |
|---------|-------------|
| 1 | Imports y setup |
| 2 | `FoundryConfig` (dataclass inmutable) |
| 3 | `FoundryConfig.from_env()` (factory + validación) |
| 4 | `FoundryEmbedder.__init__` |
| 5 | `FoundryEmbedder.embed()` |
| 6 | `FoundryEmbedder.embedding_dim()` |
| 7 | Helpers: `load_first_chunk` + `smoke_test` |
| 8 | Smoke test real con 1 chunk del corpus |
| 9 | 📋 **Síntesis**: archivo completo ensamblado |
| 10 | 🔍 **Comparación**: tu código vs el original (`%load`) |

## Cómo usar este notebook

1. Lee la celda markdown (tiene la pregunta o el contexto)
2. Tipeas tu intento en la celda de código siguiente
3. La corres — si pasa, seguimos; si falla, lo refinamos juntos
4. **No abrir el original (`src/embeddings/foundry_client.py`) hasta llegar a la celda 10** — el spoiler mata el aprendizaje 🙈

---

In [11]:
# Tu turno: escribe el primer import aquí

import json
import os
from pathlib import Path

import numpy as np
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from dotenv import load_dotenv
from openai import OpenAI


load_dotenv()

True

## 2 — `FoundryConfig`

Una clase pequeña para guardar las 2 cosas que necesitamos identificar para hablar con Foundry:

- `endpoint`: URL del recurso Azure OpenAI (v1)
- `deployment`: nombre del modelo desplegado (`text-embedding-3-small`)

> ℹ️ No guardamos `api_key` porque la auth la maneja Microsoft Entra ID via tu sesión de CLI (`az login`). Eso es estándar production.

Usamos `@dataclass` (decorador de la stdlib) para que Python escriba el `__init__` automáticamente.

In [12]:
# Tu turno: define FoundryConfig con @dataclass


from dataclasses import dataclass

@dataclass(frozen=True)
class FoundryConfig:
      endpoint: str
      deployment: str


## 3 — `from_env()` (factory method)

Queremos crear el config leyendo las 2 vars del `.env` automáticamente:

```python
config = FoundryConfig.from_env()
```

Para eso le agregamos un `@classmethod` a la clase. Primero la versión simple, después le agregamos validación.

In [13]:
# Re-define FoundryConfig agregándole el classmethod from_env()
from dataclasses import dataclass


@dataclass(frozen=True)
class FoundryConfig:
    endpoint: str
    deployment: str

    @classmethod
    def from_env(cls) -> "FoundryConfig":
        endpoint = os.getenv("AZURE_FOUNDRY_ENDPOINT", "")
        deployment = os.getenv("AZURE_FOUNDRY_EMBEDDING_DEPLOYMENT", "")
        return cls(endpoint=endpoint, deployment=deployment)


# Probémoslo
config = FoundryConfig.from_env()
print(f"endpoint:   {config.endpoint}")
print(f"deployment: {config.deployment}")


endpoint:   https://financebench-rag-eval-resource.openai.azure.com/openai/v1
deployment: text-embedding-3-small


## 4 — `FoundryEmbedder.__init__`

Ahora la clase principal — el embedder que va a producir vectores. Su `__init__`:

1. Recibe (opcionalmente) un `FoundryConfig`. Si no se lo pasas, lo crea desde `.env`.
2. Construye un **token provider** usando tu sesión `az login` (Microsoft Entra ID).
3. Instancia el cliente de OpenAI con `base_url=endpoint` y `api_key=token` (el SDK acepta el token AD como api_key).

### 🔑 ¿Qué es `get_bearer_token_provider()`?

Función helper de `azure.identity` que **envuelve una credencial en una función llamable**. Recibe:

- Una credencial (`DefaultAzureCredential()` → autodetecta tu auth: `az login`, env vars, managed identity, etc.)
- Un **scope** (string que indica qué API quieres usar — `https://cognitiveservices.azure.com/.default` cubre Azure OpenAI / Foundry)

Y devuelve una función `token_provider()` que, cuando la llamas, pide a Microsoft Entra ID un **Bearer token fresco** para ese scope. Cada llamada = token nuevo (con cache interno para no spamear al servicio).

> Por qué un provider y no un token directo: los tokens AD expiran (~1h). Tener una función te permite pedir uno nuevo on-demand sin manejar refresh tú mismo.

### 🧩 Encapsulación en 3 capas

| Capa | Qué hace | Quién la define |
|------|----------|-----------------|
| **Config** (`FoundryConfig`) | Datos: endpoint + deployment. Inmutable. | Datos puros |
| **Embedder** (`FoundryEmbedder`) | Toma el config, agrega auth, instancia el cliente. | Comportamiento |
| **Cliente** (`OpenAI`) | HTTP requests reales. Genérico (sirve para chat, embeddings, etc.). | SDK externo |

Quien use `embedder.embed([...])` no se entera de auth, endpoints ni SDK — toda esa complejidad queda escondida. Mismo patrón vas a repetir para `OpenAIEmbedder`, `BGEEmbedder`, etc.: cambia el "cómo", la interfaz (`.embed`) es la misma.

In [14]:
# Tu turno: define la clase FoundryEmbedder con su __init__

class FoundryEmbedder:
    def __init__(self, config: FoundryConfig | None = None) -> None:
        self.config = config or FoundryConfig.from_env()

        token_provider = get_bearer_token_provider(
            DefaultAzureCredential(),
            "https://cognitiveservices.azure.com/.default",
        )

        self._client = OpenAI(
            base_url=self.config.endpoint,
            api_key=token_provider(),
        )


# Probémoslo
embedder = FoundryEmbedder()
print(f"endpoint:   {embedder.config.endpoint}")
print(f"deployment: {embedder.config.deployment}")
print(f"client:     {type(embedder._client).__name__}")


endpoint:   https://financebench-rag-eval-resource.openai.azure.com/openai/v1
deployment: text-embedding-3-small
client:     OpenAI


## 5 — `embed()`

El método que realmente consume la API. Recibe textos → devuelve vectores como `np.ndarray` de shape `(n_textos, dim)`.

Pasos internos:

1. Si la lista viene vacía → devuelve ndarray vacío (caso borde).
2. Llama `self._client.embeddings.create(input=texts, model=self.config.deployment)`.
3. La respuesta trae `response.data` = lista de objetos con `.embedding` (lista de floats).
4. Convierte cada embedding a ndarray (`float32` para ahorrar memoria) y los apila en un solo array 2D con `np.stack`.

> Redefinimos la clase entera (con `__init__` + `embed`) en la siguiente celda — recuerda que es la convención del notebook.

In [15]:
# Re-define FoundryEmbedder agregando el método embed()

class FoundryEmbedder:
    def __init__(self, config: FoundryConfig | None = None) -> None:
        self.config = config or FoundryConfig.from_env()

        token_provider = get_bearer_token_provider(
            DefaultAzureCredential(),
            "https://cognitiveservices.azure.com/.default",
        )

        self._client = OpenAI(
            base_url=self.config.endpoint,
            api_key=token_provider(),
        )

    def embed(self, texts: list[str]) -> np.ndarray:
        if not texts:
            return np.empty((0, 0), dtype=np.float32)

        response = self._client.embeddings.create(
            input=texts,
            model=self.config.deployment,
        )
        vectors = [np.asarray(item.embedding, dtype=np.float32) for item in response.data]
        return np.stack(vectors, axis=0)


# Probémoslo con 2 textos sintéticos
embedder = FoundryEmbedder()
vectors = embedder.embed(["smoke test 1", "smoke test 2"])
print(f"shape:    {vectors.shape}")
print(f"dtype:    {vectors.dtype}")
print(f"vector 1:  {vectors[0,:5]}")
print(f"vector 2:  {vectors[1,:5]}")
print(f"norm L2:  {np.linalg.norm(vectors[0]):.4f}")


shape:    (2, 1536)
dtype:    float32
vector 1:  [ 0.02056885 -0.02178955 -0.02529907 -0.04284668 -0.05065918]
vector 2:  [ 0.01914978 -0.02760315 -0.02839661 -0.06463623 -0.05206299]
norm L2:  1.0001


## 6 — `embedding_dim()`

Helper que devuelve la dimensión del vector que produce el modelo (1536 para `text-embedding-3-small`).

**Por qué útil**: si vas a crear una colección en Qdrant (o cualquier vector DB), necesitas declarar la dim al crearla. Preguntar en runtime en vez de hardcodear `1536` te deja cambiar el modelo sin tocar nada más.

**Trade-off**: hace 1 llamada extra a la API (barata, ~$0.000001). Para producción podrías cachear el resultado.

In [16]:
# Re-define FoundryEmbedder agregando el método embedding_dim()

class FoundryEmbedder:
    def __init__(self, config: FoundryConfig | None = None) -> None:
        self.config = config or FoundryConfig.from_env()

        token_provider = get_bearer_token_provider(
            DefaultAzureCredential(),
            "https://cognitiveservices.azure.com/.default",
        )

        self._client = OpenAI(
            base_url=self.config.endpoint,
            api_key=token_provider(),
        )

    def embed(self, texts: list[str]) -> np.ndarray:
        if not texts:
            return np.empty((0, 0), dtype=np.float32)

        response = self._client.embeddings.create(
            input=texts,
            model=self.config.deployment,
        )
        vectors = [np.asarray(item.embedding, dtype=np.float32) for item in response.data]
        return np.stack(vectors, axis=0)

    def embedding_dim(self) -> int:
        return self.embed(["probe"]).shape[1]


# Probémoslo
embedder = FoundryEmbedder()
print(f"embedding_dim: {embedder.embedding_dim()}")


embedding_dim: 1536


## 7 — Helper `load_first_chunk()`

Función chiquita que abre un `.jsonl`, lee la primera línea y la parsea como JSON. Útil para smoke tests con datos reales sin tener que cargar el corpus entero.

Lógica:

1. Abre el archivo
2. `readline()` → trae solo la primera línea
3. `json.loads(...)` → la convierte de string a dict

Estructura esperada del chunk: `{doc_name, page_num, chunk_type, text, n_tokens, chunk_id}`.

In [17]:
# Helper para leer el primer chunk de un .jsonl

def load_first_chunk(jsonl_path: Path) -> dict:
    with jsonl_path.open("r") as f:
        line = f.readline()
    return json.loads(line)


# Probémoslo con un doc real del corpus
REPO_ROOT = Path.cwd().parent  # subimos de notebooks/ al root del repo
chunks_path = REPO_ROOT / "data" / "processed" / "chunks" / "MICROSOFT_2023_10K.jsonl"

chunk = load_first_chunk(chunks_path)
print(f"chunk_id:  {chunk['chunk_id']}")
print(f"doc_name:  {chunk['doc_name']}")
print(f"page_num:  {chunk['page_num']}")
print(f"n_tokens:  {chunk['n_tokens']}")
print(f"text[:200]: {chunk['text'][:200]}...")


chunk_id:  MICROSOFT_2023_10K_0000
doc_name:  MICROSOFT_2023_10K
page_num:  1
n_tokens:  512
text[:200]: UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
FORM 10-K
☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the Fiscal Year Ended June ...


## 8 — Smoke test real (end-to-end)

Validación con un chunk de verdad del corpus FinanceBench. La pipeline completa: `chunk → API → vector`.

Si esto funciona, ya tienes todo el módulo `foundry_client` operativo.

In [18]:
# Smoke test: cargar 1 chunk real del corpus y embebirlo

embedder = FoundryEmbedder()
chunk = load_first_chunk(chunks_path)
vectors = embedder.embed([chunk["text"]])

print(f"chunk_id:       {chunk['chunk_id']}")
print(f"doc_name:       {chunk['doc_name']}")
print(f"n_tokens:       {chunk['n_tokens']}")
print(f"vector shape:   {vectors.shape}")
print(f"vector dtype:   {vectors.dtype}")
print(f"first 5 dims:   {vectors[0, :5]}")
print(f"norm L2:        {np.linalg.norm(vectors[0]):.4f}")


chunk_id:       MICROSOFT_2023_10K_0000
doc_name:       MICROSOFT_2023_10K
n_tokens:       512
vector shape:   (1, 1536)
vector dtype:   float32
first 5 dims:   [ 0.02485657  0.01250458  0.02427673  0.0401001  -0.00170612]
norm L2:        1.0002


## 9 — 📋 Síntesis: archivo completo

Todo lo que construimos pieza por pieza, ensamblado como se vería el archivo real `foundry_client.py`.

Esta es la foto final del módulo. Si copiaras esta celda a un `.py`, tendrías un módulo importable.

In [ ]:
# =============================================================================
#  foundry_client.py — versión síntesis (todas las piezas ensambladas)
# =============================================================================

from __future__ import annotations

import json
import os
from dataclasses import dataclass
from pathlib import Path

import numpy as np
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

REPO_ROOT = Path(__file__).resolve().parents[2] if "__file__" in dir() else Path.cwd().parent
DEFAULT_CHUNKS_DIR = REPO_ROOT / "data" / "processed" / "chunks"


@dataclass(frozen=True)
class FoundryConfig:
    endpoint: str
    deployment: str

    @classmethod
    def from_env(cls) -> FoundryConfig:
        endpoint = os.getenv("AZURE_FOUNDRY_ENDPOINT", "")
        deployment = os.getenv("AZURE_FOUNDRY_EMBEDDING_DEPLOYMENT", "")
        return cls(endpoint=endpoint, deployment=deployment)


class FoundryEmbedder:
    def __init__(self, config: FoundryConfig | None = None) -> None:
        self.config = config or FoundryConfig.from_env()

        token_provider = get_bearer_token_provider(
            DefaultAzureCredential(),
            "https://cognitiveservices.azure.com/.default",
        )

        self._client = OpenAI(
            base_url=self.config.endpoint,
            api_key=token_provider(),
        )

    def embed(self, texts: list[str]) -> np.ndarray:
        if not texts:
            return np.empty((0, 0), dtype=np.float32)

        response = self._client.embeddings.create(
            input=texts,
            model=self.config.deployment,
        )
        vectors = [np.asarray(item.embedding, dtype=np.float32) for item in response.data]
        return np.stack(vectors, axis=0)

    def embedding_dim(self) -> int:
        return self.embed(["probe"]).shape[1]


def load_first_chunk(jsonl_path: Path) -> dict:
    with jsonl_path.open("r") as f:
        line = f.readline()
    return json.loads(line)


print("✅ Síntesis cargada: FoundryConfig, FoundryEmbedder, load_first_chunk listos.")


## 10 — 🔍 Comparación con el original

Cargamos el archivo `src/embeddings/foundry_client.py` del repo (el que ya está en producción y procesó los 31,216 chunks).

**Cómo funciona la magic `%load`**: cuando corres la celda, Jupyter reemplaza su contenido con el archivo apuntado. Después puedes scrollear arriba a la sección 9 (tu síntesis) y comparar lado a lado.

> Diferencias esperadas con tu versión:
> - El original todavía usa **API key** (no Microsoft Entra ID) — vale la pena migrarlo a tu approach.
> - Tiene docstrings extensos y validación en `from_env()` — robustez para producción.
> - Define helpers extra: `smoke_test()` y `smoke_test_with_real_chunk()`.

In [ ]:
# %load ../src/embeddings/foundry_client.py
"""Microsoft Foundry embedding client (provider transversal cloud).

Wrapper sobre el SDK `openai` apuntando al endpoint Azure OpenAI v1
(`https://<resource>.openai.azure.com/openai/v1/`). Carga credenciales
desde `.env` y expone una interfaz consistente con los otros embedders del
proyecto (`OpenAIEmbedder`, `BGEEmbedder`, etc.) para drop-in en el
`Eval Pipeline`.

Por qué SDK `openai` y no `azure-ai-inference`:
  - Microsoft mismo recomienda `openai` para embeddings (el endpoint del
    Project del Foundry SDK no rutea embedding requests).
  - `azure-ai-inference` está deprecated (retire 26 ago 2026).
  - Compatibilidad 1:1 con OpenAI directo (cambiar 1 var migra de Azure
    OpenAI a OpenAI sin tocar código).

Uso típico:

    from src.embeddings.foundry_client import FoundryEmbedder

    embedder = FoundryEmbedder()
    vectors = embedder.embed(["chunk de prueba", "otro chunk"])
    # vectors: np.ndarray shape (2, 1536) para text-embedding-3-small

Memoria descriptiva: https://www.notion.so/35a6ad30ca11811d96ebf4a9d7dde20b
Setup detallado:     docs/foundry_setup.md
"""

from __future__ import annotations

import json
import os
from collections.abc import Iterable
from dataclasses import dataclass
from pathlib import Path

import numpy as np
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# Repo root = 2 niveles arriba de este file (src/embeddings/foundry_client.py).
REPO_ROOT = Path(__file__).resolve().parents[2]
DEFAULT_CHUNKS_DIR = REPO_ROOT / "data" / "processed" / "chunks"


@dataclass(frozen=True)
class FoundryConfig:
    endpoint: str
    api_key: str
    deployment: str

    @classmethod
    def from_env(cls) -> FoundryConfig:
        endpoint = os.getenv("AZURE_FOUNDRY_ENDPOINT", "")
        api_key = os.getenv("AZURE_FOUNDRY_API_KEY", "")
        deployment = os.getenv("AZURE_FOUNDRY_EMBEDDING_DEPLOYMENT", "")

        missing = [
            k
            for k, v in {
                "AZURE_FOUNDRY_ENDPOINT": endpoint,
                "AZURE_FOUNDRY_API_KEY": api_key,
                "AZURE_FOUNDRY_EMBEDDING_DEPLOYMENT": deployment,
            }.items()
            if not v or "PEGA" in v
        ]
        if missing:
            raise RuntimeError(
                f"Faltan variables Foundry en .env: {missing}. Ver docs/foundry_setup.md."
            )
        return cls(endpoint=endpoint, api_key=api_key, deployment=deployment)


class FoundryEmbedder:
    """Embedder que consume modelos de embeddings desde Microsoft Foundry
    via el endpoint Azure OpenAI v1 + SDK `openai`."""

    def __init__(self, config: FoundryConfig | None = None) -> None:
        self.config = config or FoundryConfig.from_env()
        self._client = OpenAI(
            api_key=self.config.api_key,
            base_url=self.config.endpoint,
        )

    def embed(self, texts: Iterable[str]) -> np.ndarray:
        """Vectoriza una lista de textos. Devuelve ndarray shape (n, dim).

        ⚠️ Sin batch logic interna — una sola llamada por invocación. Para
        corpus grandes (≥2K inputs por request es el max de Azure OpenAI),
        orquestar el batch desde fuera con cost tracking (sub-bloque 7).
        """
        texts = list(texts)
        if not texts:
            return np.empty((0, 0), dtype=np.float32)

        response = self._client.embeddings.create(
            input=texts,
            model=self.config.deployment,
        )
        vectors = [np.asarray(item.embedding, dtype=np.float32) for item in response.data]
        return np.stack(vectors, axis=0)

    def embedding_dim(self) -> int:
        """Devuelve la dimensión del vector que produce el modelo desplegado.

        Para `text-embedding-3-small` es 1536. Para `-large` es 3072.
        Si necesitas el valor antes de la primera llamada, hardcodéalo desde
        el config; aquí lo derivamos del primer embed real para evitar
        suposiciones.
        """
        return self.embed(["probe"]).shape[1]


def load_first_chunk(jsonl_path: Path) -> dict:
    """Carga el primer chunk de un .jsonl del corpus FinanceBench.

    Estructura esperada del chunk:
        {doc_name, page_num, chunk_type, text, n_tokens, chunk_id}
    """
    with jsonl_path.open("r") as f:
        line = f.readline()
    return json.loads(line)


def smoke_test() -> dict:
    """Smoke test de conexión + 1 embedding sobre texto sintético.

    Útil para validar credenciales sin tocar el corpus. Lanza si algo falla.
    """
    embedder = FoundryEmbedder()
    sample = "smoke test from Foundry"
    vec = embedder.embed([sample])
    return {
        "endpoint": embedder.config.endpoint,
        "deployment": embedder.config.deployment,
        "input_text": sample,
        "vector_shape": tuple(vec.shape),
        "vector_dim": int(vec.shape[1]),
        "first_5_values": vec[0, :5].tolist(),
        "norm_l2": float(np.linalg.norm(vec[0])),
    }


def smoke_test_with_real_chunk(
    jsonl_path: Path | None = None,
) -> dict:
    """Smoke test sobre 1 chunk real del corpus FinanceBench.

    Por default agarra el primer chunk de `MICROSOFT_2023_10K.jsonl` (estable
    como referencia). Pasale otro path si quieres probar con un doc distinto.
    """
    path = jsonl_path or (DEFAULT_CHUNKS_DIR / "MICROSOFT_2023_10K.jsonl")
    if not path.exists():
        raise FileNotFoundError(
            f"No existe {path}. ¿Está cerrado el sub-bloque de FinanceBench Loader?"
        )

    chunk = load_first_chunk(path)
    embedder = FoundryEmbedder()
    vec = embedder.embed([chunk["text"]])

    return {
        "endpoint": embedder.config.endpoint,
        "deployment": embedder.config.deployment,
        "chunk_id": chunk["chunk_id"],
        "doc_name": chunk["doc_name"],
        "page_num": chunk["page_num"],
        "n_tokens": chunk["n_tokens"],
        "text_preview": chunk["text"][:120] + "...",
        "vector_shape": tuple(vec.shape),
        "vector_dim": int(vec.shape[1]),
        "first_5_values": vec[0, :5].tolist(),
        "norm_l2": float(np.linalg.norm(vec[0])),
    }


if __name__ == "__main__":
    import sys

    mode = sys.argv[1] if len(sys.argv) > 1 else "synthetic"
    result = smoke_test_with_real_chunk() if mode == "real" else smoke_test()
    print(json.dumps(result, indent=2))
